# 1.5.1. Iš anksto apmokytų modelių pritaikymas nuotraukai

**Liutauras Turauskis · DIPf-24 · ND1 tęsinys**

Pritaikyta pagal dėstytojo pateiktą `cnn_One_Image.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. Išsaugoti klasifikavimas, iškirptos srities analizė, objektų aptikimas ir semantinis segmentavimas. Naudojama parinkta gėlių rinkinio nuotrauka.

**Paleidimas:** Google Colab → „Runtime“ → „Run all“. Ląsteles vykdyti nuo pradžios.
Duomenys atsisiunčiami automatiškai; savo kodo, failų ir API rakto įterpti nereikia.
GPU rekomenduojamas vaizdų mokymui, bet nėra būtinas. Pirmas atsisiuntimas gali užtrukti.

**Rezultatų sąžiningumas:** šioje pateikiamoje kopijoje mokymo išvestys išvalytos.
Rodikliai, grafikai ir konkrečios išvados apskaičiuojami paleidus, o ne įrašomi iš anksto.
Parengimo aplinkoje TensorFlow mokymas nebuvo vykdytas; prieš pateikiant reikia paleisti visą užrašyną
Colab ir išsaugoti kopiją su išvestimis. Atsitiktinė sėkla mažina rezultatų svyravimą, tačiau GPU
ir bibliotekų versijos gali lemti nedidelius skirtumus.


In [ ]:
%pip -q install "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" requests pillow


In [ ]:
from pathlib import Path
import io, json, zipfile, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report)
from IPython.display import display
SEED = 42
OUT = Path('rezultatai')
OUT.mkdir(exist_ok=True)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


## 1. Pasirinkti vaizdų duomenys

Originalo CIFAR-10 / interneto nuotraukos pakeistos **Flowers Photos** rinkiniu:
`daisy`, `dandelion`, `roses`, `sunflowers`, `tulips` (5 klasės).
[TensorFlow šaltinis](https://www.tensorflow.org/tutorials/load_data/images).
Visas archyvas užima apie 218 MB. Nuotraukų autoriai ir CC BY licencijos pateikti archyvo `LICENSE.txt`.
Tai vieši pasirinkti duomenys, o ne mano paties fotografuotos nuotraukos.

Eksperimentui atsitiktinai pasirenkama iki 200 nuotraukų iš kiekvienos klasės.
Pasikartojantys vienodi failų turiniai atmetami prieš skaidymą. Mokymo, validavimo ir testo dalys:
70 %, 15 %, 15 %. Visuose trijuose vaizdų užrašynuose parinkimo taisyklės vienodos.
Panašių, bet ne identiškų nuotraukų šis patikrinimas neaptinka.


In [ ]:
from PIL import Image, ImageDraw
from collections import Counter
import tarfile
CACHE = Path('/content/nd1_duomenys') if Path('/content').exists() else Path('nd1_duomenys')
CACHE.mkdir(exist_ok=True)
archive = keras.utils.get_file(fname='flower_photos.tgz',
    origin='https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',
    cache_dir=str(CACHE.resolve()), cache_subdir='archyvai', extract=False)
DATA = CACHE / 'flower_photos'
if not DATA.exists():
    with tarfile.open(archive) as tar:
        tar.extractall(CACHE, filter='data')
classes = sorted(p.name for p in DATA.iterdir() if p.is_dir())
assert classes == ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
rng = np.random.default_rng(SEED)
rows, seen = [], set()
for label, name in enumerate(classes):
    files = sorted((DATA / name).glob('*.jpg'))
    # Atsitiktinimas atliekamas po rūšiavimo, kad failų sistemos tvarka nekeistų imties.
    files = [files[int(i)] for i in rng.permutation(len(files))]
    selected = 0
    for path in files:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen:
            continue
        seen.add(digest)
        rows.append({'path': str(path), 'relative_path': str(path.relative_to(DATA)),
                     'label': label, 'class_name': name, 'sha256': digest})
        selected += 1
        if selected == 200:
            break
images = pd.DataFrame(rows)
train_img, remaining = train_test_split(images, test_size=.30, stratify=images.label, random_state=SEED)
val_img, test_img = train_test_split(remaining, test_size=.50, stratify=remaining.label, random_state=SEED)
assert set(train_img.sha256).isdisjoint(val_img.sha256)
assert set(train_img.sha256).isdisjoint(test_img.sha256)
assert set(val_img.sha256).isdisjoint(test_img.sha256)
manifest = pd.concat([part.assign(split=name) for name, part in
    [('train', train_img), ('val', val_img), ('test', test_img)]])
manifest.drop(columns='path').to_csv(OUT / 'geliu_padalinimas.csv', index=False)
display(pd.crosstab(manifest.class_name, manifest.split))
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, name in zip(axes, classes):
    path = train_img.loc[train_img.class_name == name, 'path'].iloc[0]
    ax.imshow(Image.open(path).convert('RGB')); ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()


## 2. Du klasifikatoriai ir jų įvestys

Originalas lygino 21 tūkst. ir 1 tūkst. klasių TF Hub modelius; čia naudojami Keras
`MobileNetV2` ir `EfficientNetB0`, abu apmokyti ImageNet 1000 klasių rinkiniu.
Tai leidžia abiem taikyti tą patį patikimą `decode_predictions` žodyną.
MobileNet įvestį reikia perkelti į −1..1, o EfficientNet turi savo mastelio keitimo sluoksnį
ir gauna 0..255 reikšmes. Šiame faile svoriai **nepermokomi**: tai parengto modelio panaudojimas.
Naujos 5 gėlių klasės iš tiesų išmokomos kitame, 1.5.2 faile.


In [ ]:
photo_path = test_img.loc[test_img.class_name == 'sunflowers', 'path'].iloc[0]
photo = Image.open(photo_path).convert('RGB')
plt.figure(figsize=(6, 5)); plt.imshow(photo); plt.title('Pasirinkta testo nuotrauka: sunflowers'); plt.axis('off'); plt.show()
mobile = keras.applications.MobileNetV2(weights='imagenet')
efficient = keras.applications.EfficientNetB0(weights='imagenet')

def classify(pil_image):
    array = np.asarray(pil_image.resize((224, 224)), dtype='float32')[None]
    results = []
    for name, net, preprocessing in [
        ('MobileNetV2', mobile, keras.applications.mobilenet_v2.preprocess_input),
        ('EfficientNetB0', efficient, keras.applications.efficientnet.preprocess_input)]:
        probabilities = net.predict(preprocessing(array.copy()), verbose=0)
        # Abu modeliai jau turi softmax. Antrą kartą softmax netaikomas.
        decoded = keras.applications.imagenet_utils.decode_predictions(probabilities, top=5)[0]
        results.extend({'modelis': name, 'klasė_ImageNet': label, 'tikimybės_įvertis': float(p)}
                       for _, label, p in decoded)
    return pd.DataFrame(results)

whole_predictions = classify(photo)
display(whole_predictions)
whole_predictions.to_csv(OUT / 'isankstiniu_modeliu_top5.csv', index=False)


## 3. Iškirptos srities įtaka

Originale kadravimas keitė, ką modelis mato. Čia iškerpama centrinė nuotraukos sritis:
paliekama 60 % pločio ir aukščio. Tai nekeičia etikečių ir modelio, bet keičia įvesties kontekstą.
Lyginamas top-5 sąrašas; neskelbiama accuracy, nes rinkinio `sunflowers` klasė nėra bendras
1000 klasių ImageNet testas.


In [ ]:
w, h = photo.size
crop = photo.crop((int(.2*w), int(.2*h), int(.8*w), int(.8*h)))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(photo); axes[0].set_title('Visa nuotrauka'); axes[0].axis('off')
axes[1].imshow(crop); axes[1].set_title('Centrinė iškarpa'); axes[1].axis('off')
plt.tight_layout(); plt.show()
display(classify(crop))
# Atlaisvinama klasifikatorių atmintis prieš didesnius vaizdų analizės modelius.
del mobile, efficient
keras.backend.clear_session()
import gc
gc.collect()


## 4. Objektų aptikimas

Kaip originale, naudojamas iš anksto apmokytas Open Images SSD MobileNetV2 detektorius.
Jis grąžina ne visos nuotraukos klasę, o objektų stačiakampius ir jų pasitikėjimo įverčius.
Koordinatės pateikiamos normalizuota tvarka `(ymin, xmin, ymax, xmax)` ir padauginamos iš nuotraukos dydžio.
Rodomi tik įverčiai ≥ 0,25; jei tokių nėra, tai aiškiai parašoma.

SavedModel atsisiunčiamas pagal [oficialų TF Hub archyvų protokolą](https://www.tensorflow.org/hub/hosting)
ir įkeliamas su `tf.saved_model.load`. Taip nereikia maišyti senesnio `hub.KerasLayer` ir Keras 3.


In [ ]:
def load_saved_hub(handle, name):
    folder = CACHE / name
    if not (folder / 'saved_model.pb').exists():
        archive_path = keras.utils.get_file(fname=name + '.tar.gz',
            origin=handle + '?tf-hub-format=compressed',
            cache_dir=str(CACHE.resolve()), cache_subdir='modeliai', extract=False)
        folder.mkdir(exist_ok=True)
        if not tarfile.is_tarfile(archive_path):
            raise ValueError('Modelio šaltinis negrąžino tar archyvo. Patikrinkite modelio prieinamumą.')
        with tarfile.open(archive_path) as tar:
            tar.extractall(folder, filter='data')
    return tf.saved_model.load(str(folder))

detector = load_saved_hub('https://tfhub.dev/google/openimages_v4/ssd/mobilenet_v2/1', 'ssd_openimages')
detect = detector.signatures['default']
# Aptikimas vykdomas sumažinta nuotrauka; santykinės koordinatės tinka ir originalui.
small = photo.copy()
small.thumbnail((640, 640))
detected = detect(tf.convert_to_tensor(np.asarray(small, dtype='float32')[None] / 255.0))
boxes = detected['detection_boxes'].numpy()
scores = detected['detection_scores'].numpy()
labels = detected['detection_class_entities'].numpy()
annotated = photo.copy()
draw = ImageDraw.Draw(annotated)
found = []
for box, score, label in zip(boxes, scores, labels):
    if score < .25:
        continue
    y0, x0, y1, x1 = box
    label = label.decode('utf-8') if isinstance(label, bytes) else str(label)
    draw.rectangle((x0*w, y0*h, x1*w, y1*h), outline='red', width=3)
    draw.text((x0*w, y0*h), f'{label} {score:.2f}', fill='red')
    found.append({'objektas': label, 'įvertis': float(score)})
display(pd.DataFrame(found))
if not found:
    print('Nėra objektų, viršijančių 0,25 slenkstį; gėlės aptikimas nėra garantuotas.')
plt.figure(figsize=(8, 6)); plt.imshow(annotated); plt.axis('off'); plt.title('Objektų aptikimas'); plt.show()
annotated.save(OUT / 'objektu_aptikimas.png')
del detector, detect, detected
gc.collect()


## 5. Semantinis segmentavimas

HRNet, apmokytas ADE20K, priskiria klasę kiekvienam pikseliui. Tai skiriasi nuo objektų
stačiakampių aptikimo. Šis modelis turi 151 kanalą: foną / ignoruojamą klasę ir 150 ADE20K klasių.
Kad nebūtų klaidingai sujungti skirtingai indeksuoti klasių žodynai, pateikiami modelio klasių ID.
Modelio įvestis yra RGB 0..1; sumažinama iki 256×256, kad reikėtų mažiau atminties.
Pikselių dalis nereiškia segmentavimo tikslumo – tam reikėtų teisingų pikselių kaukių.


In [ ]:
segmenter = load_saved_hub('https://tfhub.dev/google/HRNet/ade20k-hrnetv2-w48/1', 'hrnet_ade20k')
seg_input = np.asarray(photo.resize((256, 256)), dtype='float32') / 255.0
# Šis SavedModel eksportuoja predict metodą, kaip oficialiame HRNet pavyzdyje.
seg_scores = segmenter.predict(tf.convert_to_tensor(seg_input[None]))
seg_scores = np.asarray(seg_scores)
assert seg_scores.ndim == 4 and seg_scores.shape[0] == 1
mask = np.argmax(seg_scores[0], axis=-1)
ids, counts = np.unique(mask, return_counts=True)
distribution = pd.DataFrame({'klases_ID': ids, 'pikseliu_dalis': counts / mask.size}).sort_values('pikseliu_dalis', ascending=False)
display(distribution.head(10))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(seg_input); axes[0].set_title('Įvestis')
axes[1].imshow(mask, cmap='nipy_spectral', vmin=0, vmax=150); axes[1].set_title('Kiekvieno pikselio klasė')
top_id = int(distribution.klases_ID.iloc[0])
axes[2].imshow(mask == top_id, cmap='gray'); axes[2].set_title(f'Didžiausia sritis: ID {top_id}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
np.save(OUT / 'segmentavimo_klasiu_kauke.npy', mask)
distribution.to_csv(OUT / 'segmentavimo_pikseliu_dalys.csv', index=False)
print('Klasifikavimas aprašo visą vaizdą; aptikimas randa stačiakampius; segmentavimas žymi pikselius.')


## 6. Pritaikymo išvada

Iš anksto apmokyti modeliai panaudoti naujai pasirinktai nuotraukai ir jos iškarpai.
Klasių sąrašai skirtingi: ImageNet, Open Images ir ADE20K. Todėl jų ID ir pasitikėjimų
negalima tiesiogiai lyginti tarpusavyje. Gėlė gali būti neatpažinta arba priskirta kitai klasei.
Šis pavyzdys rodo jau išmoktų reprezentacijų pritaikymą; kitame faile atliekamas tikras papildomas mokymas.

Šaltiniai: [objektų aptikimas](https://www.tensorflow.org/hub/tutorials/object_detection),
[HRNet ir jo predict metodas](https://www.tensorflow.org/hub/tutorials/hrnet_semantic_segmentation),
[Keras Applications](https://keras.io/api/applications/).
